<a href="https://colab.research.google.com/github/andinaufal120/kalimantan-frp-prediction/blob/main/autogluon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon — FRP Estimation (Separate Environment)

This notebook runs AutoGluon in an isolated venv to avoid dependency conflicts
with the main modeling environment. Outputs are saved to CSV and loaded
in the main notebook (Section 10) for cross-model comparison.

## Setup

### Imports and Constants

Same constants as main notebook to ensure consistency across experiments.
`time_limit=300` seconds per fold — AutoGluon manages its own internal
model selection within this budget.

In [ ]:
from typing import Final

import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from dateutil.relativedelta import relativedelta
import shutil
import os

PATH: Final[str] = './Dataset_ML.csv'
FIG_DIR: Final[str] = './artifacts/figures/'
REP_DIR: Final[str] = './artifacts/reports/'
SEED = 42
TRAIN_MONTHS = 12
TEST_MONTHS = 3
STEP_MONTHS = 2
TIME_LIMIT = 600  # seconds per fold

FEATURES = [
    'evi', 'ndvi_age_days', 'lst_day', 'lst_night',
    'land_cover_type', 'peat_type_class', 'ssrd', 'tp_hourly',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos',
    'vpd_kpa', 'wind_speed_m_s'
]
TARGET = 'frp_log1p'

In [ ]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

In [ ]:
ensure_dir(FIG_DIR)
ensure_dir(REP_DIR)

### Load Dataset and Reconstruct Features

Dataset loaded from the same processed CSV used in the main notebook.
`X_df` constructed as a pandas DataFrame — AutoGluon requires DataFrame input,
unlike sklearn and XGBoost which accept NumPy arrays.

In [ ]:
df = pd.read_csv(PATH)
df['acq_date'] = pd.to_datetime(df['acq_date'])
df['year'] = df['acq_date'].dt.year
df['month'] = df['acq_date'].dt.month
df['year_month'] = df['acq_date'].dt.to_period('M')

df.drop(columns=['satellite', 'is_dry_season', 'oni_index', 'dmi_index'], inplace=True)

X_df = df[FEATURES].copy()
y = df[TARGET].to_numpy()

print(f"X_df shape: {X_df.shape}")
print(f"y shape:    {y.shape}")

X_df shape: (59155, 14)
y shape:    (59155,)


### Sliding Window CV Splitter

Identical splitter to main notebook. Redefined here to keep this notebook
self-contained.

In [ ]:
def sliding_window_splits(df, date_col, train_months, test_months, step_months):
    dates = df[date_col]
    start = dates.min()
    end = dates.max()

    fold_start = start
    while True:
        train_end = fold_start + relativedelta(months=train_months)
        test_end = train_end + relativedelta(months=test_months)

        if test_end > end:
            break

        train_idx = df.index[(dates >= fold_start) & (dates < train_end)].to_numpy()
        test_idx = df.index[(dates >= train_end) & (dates < test_end)].to_numpy()

        if len(train_idx) > 0 and len(test_idx) > 0:
            yield train_idx, test_idx

        fold_start += relativedelta(months=step_months)

splits = list(sliding_window_splits(
    df, 'acq_date', TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS
))

print(f"Total folds: {len(splits)}")

Total folds: 11


## 9. AutoGluon

### 9.1 Predictor Configuration

AutoGluon is given a fixed time budget of 300 seconds per fold.
Preset set to `'medium_quality'` as a practical starting point —
balances model diversity with computational feasibility across 11 folds.
Predictor trained on a pandas DataFrame with `frp_log1p` as the label column.
Predictor directory deleted after each fold to manage disk space.

### 9.2 Per-Fold Training and Evaluation

Same sliding window folds as main notebook.
After each fold, best model name extracted via `predictor.get_model_best()`,
metrics computed via `metric_function`, and predictor deleted from disk.

In [ ]:
def metric_function(y_true, y_pred):
    y_true_mw = np.expm1(y_true)
    y_pred_mw = np.expm1(y_pred)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mae_mw = np.mean(np.abs(y_true_mw - y_pred_mw))
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

    return rmse, mae, mae_mw, r2

In [ ]:
ag_results = []
leaderboards = []

for i, (train_idx, test_idx) in enumerate(splits):
    fold_path = f'./ag_fold_{i+1}/'

    train_df = X_df.iloc[train_idx].copy()
    train_df[TARGET] = y[train_idx]

    test_df = X_df.iloc[test_idx].copy()
    y_test = y[test_idx]

    predictor = TabularPredictor(
        label=TARGET,
        path=fold_path,
        problem_type='regression',
        verbosity=0
    ).fit(
        train_data=train_df,
        time_limit=TIME_LIMIT,
        presets='medium_quality'
    )

    y_pred = predictor.predict(test_df).to_numpy()
    best_model = predictor.model_best
    rmse, mae, mae_mw, r2 = metric_function(y_test, y_pred)

    ag_results.append({
        'fold': i + 1,
        'best_model': best_model,
        'rmse_log1p': rmse,
        'mae_log1p': mae,
        'mae_mw': mae_mw,
        'r2': r2,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
    })

    leaderboard = predictor.leaderboard(silent=True)
    leaderboard.insert(0, 'fold', i + 1)
    leaderboards.append(leaderboard)

    shutil.rmtree(fold_path)
    print(f"Fold {i+1:2d} — RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, MAE(MW): {mae_mw:.2f} | Best: {best_model}")

/home/yamaym/Documents/dev/python/kalimantan-frp-prediction/.venv-gluon/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold  1 — RMSE: 0.5551, MAE: 0.4093, R²: 0.4035, MAE(MW): 2.94 | Best: WeightedEnsemble_L2


/home/yamaym/Documents/dev/python/kalimantan-frp-prediction/.venv-gluon/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold  2 — RMSE: 0.5201, MAE: 0.4052, R²: 0.4081, MAE(MW): 2.76 | Best: WeightedEnsemble_L2


/home/yamaym/Documents/dev/python/kalimantan-frp-prediction/.venv-gluon/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


...

Fold  9 — RMSE: 0.5390, MAE: 0.3992, R²: 0.4339, MAE(MW): 2.96 | Best: WeightedEnsemble_L2


/home/yamaym/Documents/dev/python/kalimantan-frp-prediction/.venv-gluon/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 10 — RMSE: 0.6993, MAE: 0.5335, R²: 0.2209, MAE(MW): 6.42 | Best: WeightedEnsemble_L2


/home/yamaym/Documents/dev/python/kalimantan-frp-prediction/.venv-gluon/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 11 — RMSE: 0.6909, MAE: 0.5191, R²: 0.2381, MAE(MW): 6.14 | Best: WeightedEnsemble_L2


### 9.3 Results Summary

Per-fold and aggregate (mean ± std) metrics printed.
Best model selected by AutoGluon per fold reported alongside metrics —
equivalent of `best_params` in RF and XGBoost results.

In [ ]:
ag_df = pd.DataFrame(ag_results)

print("\nPer-fold results:")
print(ag_df[['fold', 'best_model', 'rmse_log1p', 'mae_log1p', 'mae_mw', 'r2', 'train_size', 'test_size']].to_string(index=False))

print("\nAggregate (mean ± std):")
for metric in ['rmse_log1p', 'mae_log1p', 'mae_mw', 'r2']:
    mean = ag_df[metric].mean()
    std = ag_df[metric].std()
    print(f"  {metric:15s}: {mean:.4f} ± {std:.4f}")

leaderboard_df = pd.concat(leaderboards, ignore_index=True)
ensure_dir(REP_DIR)
leaderboard_df.to_csv(f'{REP_DIR}ag_leaderboard.csv', index=False)
print(f"Leaderboard saved to {REP_DIR}ag_leaderboard.csv")

ag_df.to_csv(f'{REP_DIR}ag_results.csv', index=False)
print(f"\nSaved to {REP_DIR}ag_results.csv")


Per-fold results:
 fold          best_model  rmse_log1p  mae_log1p   mae_mw       r2  train_size  test_size
    1 WeightedEnsemble_L2    0.555149   0.409313 2.938769 0.403467        4485        399
    2 WeightedEnsemble_L2    0.520065   0.405236 2.761000 0.408052        4572        918
    3 WeightedEnsemble_L2    0.650403   0.488622 4.933742 0.321833        4564       3405
    4 WeightedEnsemble_L2    0.659511   0.497931 4.900070 0.273647        5388      27894
    5 WeightedEnsemble_L2    0.606228   0.461631 4.067617 0.354446       15872      29063
    6 WeightedEnsemble_L2    0.612379   0.471210 3.648755 0.368740       42585       1268
    7 WeightedEnsemble_L2    0.462247   0.348564 2.093289 0.510588       43391        607
    8 WeightedEnsemble_L2    0.424240   0.322618 1.659895 0.521572       43452        647
    9 WeightedEnsemble_L2    0.538962   0.399210 2.961849 0.433927       43540        795
   10 WeightedEnsemble_L2    0.699257   0.533503 6.424731 0.220948       42684   

## 10. AutoGluon Best Quality

In [ ]:
ag_results_bq = []
leaderboards_bq = []

for i, (train_idx, test_idx) in enumerate(splits):
    fold_path = f'./ag_fold_bq_{i+1}/'

    train_df = X_df.iloc[train_idx].copy()
    train_df[TARGET] = y[train_idx]

    test_df = X_df.iloc[test_idx].copy()
    y_test = y[test_idx]

    predictor = TabularPredictor(
        label=TARGET,
        path="/tmp/autogluon",
        problem_type='regression',
        verbosity=0
    ).fit(
        train_data=train_df,
        time_limit=2600,
        presets='best_quality'
    )

    y_pred = predictor.predict(test_df).to_numpy()
    best_model = predictor.model_best
    rmse, mae, mae_mw, r2 = metric_function(y_test, y_pred)

    ag_results_bq.append({
        'fold': i + 1,
        'best_model': best_model,
        'rmse_log1p': rmse,
        'mae_log1p': mae,
        'mae_mw': mae_mw,
        'r2': r2,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
    })

    leaderboard = predictor.leaderboard(silent=True)
    leaderboard.insert(0, 'fold', i + 1)
    leaderboards_bq.append(leaderboard)

    shutil.rmtree("/tmp/autogluon")
    print(f"Fold {i+1:2d} — RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, MAE(MW): {mae_mw:.2f} | Best: {best_model}")

# Save
ag_df_bq = pd.DataFrame(ag_results_bq)
ag_df_bq.to_csv(f'{REP_DIR}ag_results_best_quality.csv', index=False)

ag_lb_bq = pd.concat(leaderboards_bq, ignore_index=True)
ag_lb_bq.to_csv(f'{REP_DIR}ag_leaderboard_best_quality.csv', index=False)

print("\n--- best_quality Aggregate ---")
print(f"RMSE : {ag_df_bq['rmse_log1p'].mean():.4f} ± {ag_df_bq['rmse_log1p'].std():.4f}")
print(f"MAE  : {ag_df_bq['mae_log1p'].mean():.4f} ± {ag_df_bq['mae_log1p'].std():.4f}")
print(f"MAE(MW): {ag_df_bq['mae_mw'].mean():.4f} ± {ag_df_bq['mae_mw'].std():.4f}")
print(f"R²   : {ag_df_bq['r2'].mean():.4f} ± {ag_df_bq['r2'].std():.4f}")

Fold  1 — RMSE: 0.5318, MAE: 0.3847, R²: 0.4525, MAE(MW): 2.83 | Best: WeightedEnsemble_L3
Fold  2 — RMSE: 0.5007, MAE: 0.3858, R²: 0.4512, MAE(MW): 2.62 | Best: WeightedEnsemble_L3
Fold  3 — RMSE: 0.6365, MAE: 0.4754, R²: 0.3505, MAE(MW): 4.81 | Best: WeightedEnsemble_L2
Fold  4 — RMSE: 0.6376, MAE: 0.4798, R²: 0.3212, MAE(MW): 4.79 | Best: WeightedEnsemble_L3
Fold  5 — RMSE: 0.5995, MAE: 0.4529, R²: 0.3687, MAE(MW): 3.97 | Best: WeightedEnsemble_L3
Fold  6 — RMSE: 0.6064, MAE: 0.4661, R²: 0.3810, MAE(MW): 3.63 | Best: WeightedEnsemble_L2
Fold  7 — RMSE: 0.4598, MAE: 0.3471, R²: 0.5158, MAE(MW): 2.08 | Best: WeightedEnsemble_L2
Fold  8 — RMSE: 0.4214, MAE: 0.3188, R²: 0.5279, MAE(MW): 1.64 | Best: WeightedEnsemble_L2
Fold  9 — RMSE: 0.5327, MAE: 0.3957, R²: 0.4470, MAE(MW): 2.92 | Best: WeightedEnsemble_L2
Fold 10 — RMSE: 0.6978, MAE: 0.5314, R²: 0.2241, MAE(MW): 6.40 | Best: WeightedEnsemble_L2



KeyboardInterrupt

